In [1]:
from importlib.util import find_spec
from importlib.metadata import version

# In case you want to restart + run all, we carefully only install the packages when missing or at wrong versions
if not find_spec("torch") or version("torch") != "2.13.0.dev20260518+cpu":
  print("Installing torch...")
  ! pip install --force-reinstall --pre torch==2.13.0.dev20260518 --quiet --progress-bar off --index-url https://download-r2.pytorch.org/whl/nightly/cpu/  && echo "Installed torch"

if not find_spec("torchtitan"):
  print("Installing torchtitan...")
  ! pip install git+https://github.com/pytorch/torchtitan.git --force-reinstall --quiet --progress-bar off  && echo "Installed torchtitan."

if not find_spec("titan_demo"):
  print("Installing titan-demo...")
  ! pip install git+https://github.com/fegin/titan-demo.git --force-reinstall --quiet --progress-bar off && echo "Installed titan-demo."

Installing torch...
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
torchaudio 2.10.0+cpu requires torch==2.10.0, but you have torch 2.13.0.dev20260518+cpu which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.4.0 which is incompatible.
torchvision 0.25.0+cpu requires torch==2.10.0, but you have torch 2.13.0.dev20260518+cpu which is incompatible.
datasets 4.0.0 requires fsspec[http]<=2025.3.0,>=2023.1.0, but you have fsspec 2026.4.0 which is incompatible.
Installed torch
Installing torchtitan...
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of th

# TorchTitan Parallelism Memory Explorer

Pick a parallelism config, see whether it fits in GPU memory -- without
needing real GPUs. The model runs as FakeTensors and memory is reported by
`torch.distributed._tools.fsdp2_mem_tracker` for one full training step
(forward + backward + AdamW.step).

What gets measured per rank:
- Sharded / unsharded parameters and gradients
- Activations and backward temporaries
- All-gather and reduce-scatter buffers
- AdamW optimizer states

The verdict sums the peak across all devices in the snapshot (cuda + cpu),
since those bytes all sit in one rank's memory footprint.

## Three scenarios

| Scenario | Model      | World     | Per-GPU memory |
|----------|------------|-----------|----------------|
| 1        | Llama3 8B  | 8 GPUs    | 40 GiB         |
| 2        | Llama3 70B | 128 GPUs  | 80 GiB         |
| 3        | Llama3 405B| 1024 GPUs | 80 GiB         |

## What's compared against a real run

See the project README ("What the memory estimate represents") for how
the eager-mode estimate differs from a real `torch.compile` run and how
the `OptState` rewrite works.

## Limitations

- Pipeline Parallel (`pp > 1`) is not modeled by
  `parallelize_fake_model` today; SPMD only (TP + CP + FSDP/HSDP).
- The widgets enforce `tp * cp * dp_replicate <= world_size` and that
  `seq_len` is divisible by `tp * 2 * cp`; `dp_shard` is set
  automatically from the leftover ranks.


## Setup

In [ ]:
import torch
import ipywidgets as widgets
from IPython.display import display
from torch.distributed.tensor import DTensor

from titan_demo import (
    build_fake_model,
    estimate_memory,
    format_memory_estimate,
    make_model_spec,
    make_parallel_dims,
    parallelize_fake_model,
)


# AdamW keeps three fp32 values per parameter: master weight,
# first moment (exp_avg), second moment (exp_avg_sq).
_ADAMW_BYTES_PER_PARAM = 12


def _local_param_count(model):
    """Per-rank parameter count, summing local storage of each DTensor."""
    total = 0
    for p in model.parameters():
        total += p.to_local().numel() if isinstance(p, DTensor) else p.numel()
    return total


def _correct_optstate(snap, model):
    """Replace the tracker's OptState with a deterministic value.

    FSDPMemTracker under FakeTensorMode reports OptState on the cpu
    device and at a size that does not match a real run: AdamW state
    creation does not always preserve DTensor sharding under fake mode.
    We replace it with ``local_params * 12`` (AdamW master + exp_avg +
    exp_avg_sq, all fp32) and attribute it to the rank's compute
    device. Under FakeTensorMode that device is reported as cpu (it
    would be cuda in real training); we just reuse the device that
    already holds the other categories.
    """
    corrected = _local_param_count(model) * _ADAMW_BYTES_PER_PARAM

    compute_dev = next((d for d in snap if d.type == "cuda"), None)
    if compute_dev is None:
        compute_dev = next(iter(snap))

    for dev in list(snap):
        breakdown = snap[dev]
        for key in list(breakdown):
            name = key.value if hasattr(key, "value") else str(key)
            if name == "OptState":
                old = breakdown.pop(key)
                breakdown["Total"] = breakdown.get("Total", 0) - old
        if dev != compute_dev and breakdown.get("Total", 0) == 0:
            del snap[dev]

    snap[compute_dev]["OptState"] = corrected
    snap[compute_dev]["Total"] = snap[compute_dev].get("Total", 0) + corrected
    return snap


def per_rank_gib(snap):
    """Sum totals across all devices to get the per-rank peak in GiB."""
    return sum(d.get("Total", 0) for d in snap.values()) / (1024 ** 3)


def verdict(used_gib, budget_gib):
    pct = used_gib / budget_gib * 100
    if used_gib > budget_gib:
        return f"OOM: need {used_gib:.2f} GiB, have {budget_gib} GiB ({pct:.0f}%)"
    if pct >= 95:
        return f"NEAR OOM: {used_gib:.2f} / {budget_gib} GiB ({pct:.1f}%)"
    return f"FITS: {used_gib:.2f} / {budget_gib} GiB ({pct:.1f}%)"


def analyze(*, flavor, world_size, gpu_mem_gib,
            tp, cp, dp_replicate, batch_size, seq_len):
    if torch.distributed.is_initialized():
        # We must reinitialize distributed when world size changes in simulation
        torch.distributed.destroy_process_group()
    # CP requires FlexAttention. SDPA is fine (and slightly faster to
    # build) when cp == 1.
    attn_backend = "flex" if cp > 1 else "sdpa"
    spec = make_model_spec(flavor, seq_len=seq_len, attn_backend=attn_backend)
    model, fm = build_fake_model(spec.model, dtype=torch.bfloat16)
    pd = make_parallel_dims(
        world_size=world_size,
        tp=tp,
        cp=cp,
        dp_replicate=dp_replicate,
    )
    parallelize_fake_model(model, parallel_dims=pd)
    snap = estimate_memory(
        model, fm, pd, batch_size=batch_size, seq_len=seq_len,
    )
    _correct_optstate(snap, model)

    used = per_rank_gib(snap)
    global_batch = batch_size * pd.dp_replicate * pd.dp_shard
    print(f"Llama3 {flavor} on {world_size} GPUs x {gpu_mem_gib} GiB")
    print(f"  tp={tp}, cp={cp}, dp_replicate={dp_replicate}, dp_shard={pd.dp_shard}")
    print(f"  per-rank batch={batch_size}, seq_len={seq_len}")
    print(f"  global batch={global_batch}")
    print()
    print(f"Verdict: {verdict(used, gpu_mem_gib)}")
    print()
    print(format_memory_estimate(snap, units="GiB"))


def _validate(scenario, *, tp, cp, dp_replicate, seq_len):
    """Return an error message string, or None if config is valid."""
    world = scenario["world_size"]
    if tp * cp * dp_replicate > world:
        return (
            f"tp * cp * dp_replicate = {tp * cp * dp_replicate} > "
            f"world_size = {world}"
        )
    if world % (tp * cp * dp_replicate) != 0:
        return (
            f"world_size={world} not divisible by "
            f"tp * cp * dp_replicate = {tp * cp * dp_replicate} "
            f"(no integer dp_shard fits)"
        )
    # seq_len must be divisible by tp * 2 * cp (SP needs tp, CP needs 2*cp).
    divisor = tp * 2 * cp
    if seq_len % divisor != 0:
        return (
            f"seq_len={seq_len} not divisible by tp * 2 * cp = {divisor}"
        )
    return None


## Scenario 1: Llama3 8B on 8 GPUs (40 GiB each)

Total system memory: 8 x 40 = 320 GiB.

Rough budget:
- 8B params in bf16 = ~15 GiB (replicated)
- AdamW state (fp32 momentum + variance + master weight) = ~89 GiB total
  -> ~11 GiB per rank under FSDP=8
- Activations grow with `batch_size` and `seq_len`; TP shrinks them in
  the heads dim, CP shrinks them in the sequence dim

Things to try:
- Pure FSDP (`tp=1, cp=1`): does the leftover fit?
- Pure TP (`tp=8, cp=1`): no FSDP, every rank keeps the full param state -- watch it OOM
- TP + CP (`tp=2, cp=2`): mixes intra-node TP with seq-sharded CP
- HSDP (`dp_replicate=2`): replicate across 2 groups of 4-rank FSDP shards; more redundant memory than pure FSDP
- Increase `batch_size` or `seq_len` until you fall off the cliff


In [ ]:
SCENARIO_1 = dict(
    flavor='8B',
    world_size=8,
    gpu_mem_gib=40,
)


@widgets.interact_manual(
    tp=widgets.Dropdown(options=[1, 2, 4, 8], value=1, description="tp"),
    cp=widgets.Dropdown(options=[1, 2, 4], value=1, description="cp"),
    dp_replicate=widgets.Dropdown(
        options=[1, 2], value=1, description="dp_replicate"
    ),
    batch_size=widgets.IntSlider(
        min=1, max=4, value=1, description="batch_size"
    ),
    seq_len=widgets.Dropdown(
        options=[2048, 4096, 8192], value=8192, description="seq_len"
    ),
)
def run_scenario_1(tp, cp, dp_replicate, batch_size, seq_len):
    err = _validate(
        SCENARIO_1,
        tp=tp, cp=cp, dp_replicate=dp_replicate, seq_len=seq_len,
    )
    if err:
        print(err)
        return
    analyze(
        **SCENARIO_1,
        tp=tp,
        cp=cp,
        dp_replicate=dp_replicate,
        batch_size=batch_size,
        seq_len=seq_len,
    )


## Scenario 2: Llama3 70B on 128 GPUs (80 GiB each)

Total system memory: 128 x 80 = 10240 GiB. Sounds like a lot until you remember each rank still needs to hold its slice of:
- 70B params in bf16 = ~130 GiB
- AdamW state (fp32) = ~782 GiB
- Per-rank activations for `dim=8192`, `n_layers=80`

Things to try:
- `tp=8` (intra-node TP): the standard choice for 70B
- `tp=1`: pure FSDP across 128 ranks; do activations push you over?
- `dp_replicate=2`: HSDP with 2 replicas of 64-shard FSDP groups
- Long context with CP: bump `seq_len` to 16384 with `tp=8, cp=2`


In [ ]:
SCENARIO_2 = dict(
    flavor='70B',
    world_size=128,
    gpu_mem_gib=80,
)


@widgets.interact_manual(
    tp=widgets.Dropdown(options=[1, 2, 4, 8], value=8, description="tp"),
    cp=widgets.Dropdown(options=[1, 2, 4, 8], value=1, description="cp"),
    dp_replicate=widgets.Dropdown(
        options=[1, 2, 4], value=1, description="dp_replicate"
    ),
    batch_size=widgets.IntSlider(
        min=1, max=4, value=1, description="batch_size"
    ),
    seq_len=widgets.Dropdown(
        options=[2048, 4096, 8192, 16384], value=8192, description="seq_len"
    ),
)
def run_scenario_2(tp, cp, dp_replicate, batch_size, seq_len):
    err = _validate(
        SCENARIO_2,
        tp=tp, cp=cp, dp_replicate=dp_replicate, seq_len=seq_len,
    )
    if err:
        print(err)
        return
    analyze(
        **SCENARIO_2,
        tp=tp,
        cp=cp,
        dp_replicate=dp_replicate,
        batch_size=batch_size,
        seq_len=seq_len,
    )


## Scenario 3: Llama3 405B on 1024 GPUs (80 GiB each)

Total system memory: 1024 x 80 = 81920 GiB.

Rough budget:
- 405B params in bf16 = ~754 GiB
- AdamW state (fp32) = ~4527 GiB
- `dim=16384`, `n_layers=126`, `hidden_dim=53248` -> activations are huge

Note: this scenario *cannot* use pipeline parallel (PP) -- `titan_demo`'s `parallelize_fake_model` only wires up TP + CP + FSDP/HSDP today, and 405B training in production typically also uses PP. Treat the numbers here as "the SPMD-only lower bound".

The widget starts at `tp=8, cp=1, seq_len=2048` so the first run fits. Things to try:
- Bump `seq_len` to 4096, 8192, then 16384 -- which step makes you OOM?
- Add CP: `tp=8, cp=2, seq_len=8192` shards the sequence in half
- Drop `tp` to 4 or 2 -- how quickly do activations + temporaries blow up?
- `dp_replicate=2` with `tp=8`: HSDP for better all-reduce overlap, at the cost of replicating params
- For very long context, expect to need PP and/or activation checkpointing (neither is exposed by `titan_demo` today, so the numbers here are an SPMD-only lower bound)


In [ ]:
SCENARIO_3 = dict(
    flavor='405B',
    world_size=1024,
    gpu_mem_gib=80,
)


@widgets.interact_manual(
    tp=widgets.Dropdown(options=[1, 2, 4, 8], value=8, description="tp"),
    cp=widgets.Dropdown(options=[1, 2, 4, 8], value=1, description="cp"),
    dp_replicate=widgets.Dropdown(
        options=[1, 2, 4, 8], value=1, description="dp_replicate"
    ),
    batch_size=widgets.IntSlider(
        min=1, max=4, value=1, description="batch_size"
    ),
    seq_len=widgets.Dropdown(
        options=[2048, 4096, 8192, 16384], value=2048, description="seq_len"
    ),
)
def run_scenario_3(tp, cp, dp_replicate, batch_size, seq_len):
    err = _validate(
        SCENARIO_3,
        tp=tp, cp=cp, dp_replicate=dp_replicate, seq_len=seq_len,
    )
    if err:
        print(err)
        return
    analyze(
        **SCENARIO_3,
        tp=tp,
        cp=cp,
        dp_replicate=dp_replicate,
        batch_size=batch_size,
        seq_len=seq_len,
    )
